In [1]:
from email.mime import text

from fastwarc.warc import ArchiveIterator, WarcRecordType
from cs336_data.data_quality import *
import fasttext

## Data preparation

In [3]:
fin = open('../example.warc.wet.gz', 'rb')
iter = ArchiveIterator(fin)

total_count = 0
count = 0
threshold = 0.6
with open("./data/quality_classifier_train_negative.txt", "w") as fout:
    for record in iter:
        total_count += 1
        text = extract_text_from_html_bytes(record.reader.read())
        text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
        text = re.sub(r"\s+", " ", text).strip()
        lang, lang_conf = identify_language(text)
        if lang == "en" and lang_conf >= threshold:
            count += 1
            fout.write(f"__label__cc {text}\n")
print(f"Total number of records: {total_count}")
print(f"Number of English records with confidence >= {threshold}: {count}")

Total number of records: 20596
Number of English records with confidence >= 0.6: 7209


In [ ]:
import timeit

gstart = timeit.default_timer()

# enwiki extraction executed in terminal
# uv run python -m wikiextractor.WikiExtractor   cs336_data/data/enwiki-20260701-pages-articles-multistream15.xml-p14324603p15824602.bz2   -o - --text > cs336_data/data/enwiki_20260701_extracted.txt

fin = open("data/enwiki_20260701_extracted.txt", "r")
fout = open("data/enwiki_20260701_extracted_filtered.txt", "w")

total_count = 0
count = 0
threshold = 0.6
for text in fin:
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"\s+", " ", text).strip()
    lang, lang_conf = identify_language(text)
    if gopher_quality_filter(text) and lang == "en" and lang_conf >= threshold:
        fout.write(f"{text}\n")
        count += 1
    total_count += 1

fin.close()
fout.close()

print(f"Total Record Number: {total_count}")
print(f"Total Filtered Record Number: {count}")

fin = open("data/enwiki_20260701_extracted_filtered.txt", "rb")

with open("data/quality_classifier_train_positive.txt", "w") as fout:
    for line in fin:
        fout.write(f"__label__wiki {line}\n")
fin.close()

gstop = timeit.default_timer()
print(f"Total Execution Time: {(gstop - gstart)/60:2f} minutes")

Total Record Number: 2106131
Total Filtered Record Number: 331693
Total Execution Time: 2.257382 minutes


In [5]:
fin1 = open("./data/quality_classifier_train_positive.txt", "r")
fin2 = open("./data/quality_classifier_train_negative.txt", "r")
fout = open("./data/quality_classifier_train.txt", "w")

while True:
    line1 = fin1.readline()
    line2 = fin2.readline()
    if not line1 and not line2:
        break
    if line1:
        fout.write(line1)
    if line2:
        fout.write(line2)

fin1.close()
fin2.close()
fout.close()

In [6]:
import fasttext

model = fasttext.train_supervised(
    input="data/quality_classifier_train.txt",  # Path to your training file
    lr=0.5,                                     # Learning rate (default is 0.1)
    epoch=25,                                   # Number of epochs (default is 5)
    wordNgrams=2,                               # Use bigrams to capture word order (default is 1)
    dim=100                                     # Size of word vectors (default is 100)
)

model.save_model("text_classifier.bin")

# 5. Load a saved model later
loaded_model = fasttext.load_model("text_classifier.bin")

# Sanity check
samples, precision, recall = loaded_model.test("data/quality_classifier_train.txt")
print(f"Evaluated {samples} samples | Precision: {precision:.4f} | Recall: {recall:.4f}")


Read 40M words
Number of words:  1688962
Number of labels: 2
Progress:  99.8% words/sec/thread: 1609877 lr:  0.000911 avg.loss:  0.003500 ETA:   0h 0m 0s

Evaluated 338902 samples | Precision: 1.0000 | Recall: 1.0000


Progress: 100.0% words/sec/thread: 1609004 lr:  0.000000 avg.loss:  0.003494 ETA:   0h 0m 0s
